# Leaks, then two features

Fresh and Good is the same fictional Bangkok grocery as Day 1. The table in this folder is the Day 1 table plus two columns that were built from July. The Day 1 file is unchanged.

`churned` and `next_month_spend_thb` are in the public file. There is no hidden test. You compare models on a validation split of this file.

June features stop on 30 June 2026. A column that uses July is a leak. Drop it before you fit.


In [ ]:
import sys
from pathlib import Path

here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    marker = candidate / "lib" / "leaks.py"
    if marker.is_file() and (candidate / "app.py").is_file():
        root = candidate
        break
if root is None:
    raise FileNotFoundError("Start Jupyter from 02-segments-and-recommendations.")
lib = root / "lib"
if str(lib) not in sys.path:
    sys.path.insert(0, str(lib))
ROOT = root
print("module", ROOT)


In [ ]:
import pandas as pd

from leaks import DAY2_CSV, safe_feature_columns

frame = pd.read_csv(DAY2_CSV)
print(frame.shape)
print(list(frame.columns))
print("safe features", safe_feature_columns(frame.columns))


## Find the two July columns

Read `data/DATA_DICTIONARY.md`. Then look at how strongly each numeric column lines up with `churned`. A perfect line is a warning, not a good feature.


In [ ]:
numeric = frame.select_dtypes("number").drop(columns=["churned"])
corr = numeric.corrwith(frame["churned"]).sort_values(key=lambda s: s.abs(), ascending=False)
print(corr.round(3).to_string())


In [ ]:
leak_columns = None  # two column names that use July

if leak_columns is None:
    raise NotImplementedError("Name the two July columns.")
print("drop", leak_columns)


## Add two June features

`orders_per_week` is `orders_90d / 13`. `recency_over_45` is 1 when `days_since_last_order` is 45 or more, else 0. Both are known on 30 June.


In [ ]:
featured = frame.copy()
featured["orders_per_week"] = None
featured["recency_over_45"] = None

if featured["orders_per_week"].iloc[0] is None or featured["recency_over_45"].iloc[0] is None:
    raise NotImplementedError("Fill orders_per_week and recency_over_45.")


## Compare on validation

The next cell fits a logistic regression three times. The split is 75 percent train and 25 percent validation, seed 42, stratified on `churned`. The higher score is not a hidden-test score.


In [ ]:
from rfm import BASE_FEATURES, validation_auc

print("with July columns", round(validation_auc(featured, [*BASE_FEATURES, *leak_columns]), 3))
print("June features", round(validation_auc(featured, BASE_FEATURES), 3))
print("June plus two features", round(validation_auc(featured, [*BASE_FEATURES, "orders_per_week", "recency_over_45"]), 3))


Write one sentence under this cell: did the July columns inflate the validation score, and did the two new features beat the June features?
